In [2]:
library(kSamples)
library(glmnet)
library(ppcor)

Loading required package: SuppDists

Loading required package: Matrix

Loaded glmnet 4.1-8

Loading required package: MASS



In [ ]:
preprocess <- function(df){
  
  NAIdx <- is.na(apply(df,1,sum))
  df <- df[!NAIdx,]
  
  totDATA <- df[,1:dim(df)[2]-1]
  timeline <- df[,dim(df)[2]]
  DATA.time <- sort(unique(timeline))
  DATA.num_time_points <- length(DATA.time)
  DATA.totDATA <- matrix(ncol = dim(df)[2]-1)
  DATA.timeline <- vector()
  
  for (k in 1:DATA.num_time_points) {
    I <- which(timeline==DATA.time[k])
    DATA.totDATA <- rbind(DATA.totDATA,as.matrix(totDATA[I,]))
    DATA.timeline <- c(DATA.timeline,timeline[I])
  }
  DATA.totDATA <- DATA.totDATA[-1,]
  DATA.totDATA[is.na(DATA.totDATA)] <- 0
  DATA.numGENES <- dim(DATA.totDATA)[2]
  
  
  DATA.genes <- colnames(df)[1:dim(df)[2]-1]
  
  DATA.singleCELLdata <- by(DATA.totDATA,DATA.timeline,identity)
  DATA <- list(time=DATA.time,num_time_points=DATA.num_time_points,
               totDATA=DATA.totDATA,timeline=DATA.timeline,numGENES=DATA.numGENES,
               genes=DATA.genes,singleCELLdata=DATA.singleCELLdata)
  return(DATA)
}

In [ ]:
SINCERITITES <- dget("/home/chenxufeng/picb_cxf/Beeline-master/SINCERITIES-R_v2.0/SINCERITIES_functions/SINCERITIES.R")

In [92]:
# Input_dir = "/home/chenxufeng/picb_cxf/Beeline-master/inputs/PMID36973557_NatBiotechnol2023_T-cell-depleted/test_assoc_fdr1e-3_A0.5/"
# Output_dir = "/home/chenxufeng/picb_cxf/Data/PMID36973557_NatBiotechnol2023_T-cell-depleted/benchmark/240704/net/"
# lineages = c("Mono")

Input_dir = "/home/chenxufeng/picb_cxf/Beeline-master/inputs/PMID36973557_NatBiotechnol2023_CD34/test_assoc_fdr1e-3_A0.3/"
Output_dir = "/home/chenxufeng/picb_cxf/Data/PMID36973557_NatBiotechnol2023_CD34/benchmark/240921/net/"
lineages = c("Mega")

In [95]:
for (lin in lineages) {

    ExprMatrix = t(read.csv(paste0(Input_dir, lin, "/ExpressionData.csv"), header = T, row.names = 1, sep=","))
    Pseudotime = read.csv(paste0(Input_dir, lin, "/PseudoTime.csv"), header = T, row.names = 1, sep=",")

    sds <- apply(ExprMatrix, 2, sd)
    ExprMatrix<- ExprMatrix[, which(sds > quantile(sds, prob = 0.2))]

    sorted_indices <- order(Pseudotime$palantir_pseudotime)
    bin_indices <- cut(seq_along(sorted_indices), breaks = 10, labels = FALSE) - 1

    Pseudotime$time <- NA
    Pseudotime$time[sorted_indices] <- bin_indices

    DATA <- preprocess(cbind(ExprMatrix, Pseudotime$time))

    # Calculate the correlation matrix

    SIGN <- 0
    result <- SINCERITITES(DATA,distance=1,method = 1,noDIAG = 0,SIGN = SIGN)

    adj_matrix <- result$adj_matrix
    dimnames(adj_matrix) <- list(DATA$genes,DATA$genes)

    edge_df <- melt(adj_matrix, varnames = c("TF", "Target"), value.name = "Score")

    write.csv(edge_df, paste0(Output_dir, "SINCERITITES_", lin, ".csv"), row.names = F)
}